In [ ]:
#autor: Oliwier Sobczak

In [2]:
pip install pymongo

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pymongo import MongoClient
klient = MongoClient("mongodb://localhost:27017/")
db = klient["Sobczak_Oliwier"]

In [4]:
import os
os.getcwd()

'C:\\Users\\xXPIO\\Documents\\Sobczak_Oliwier'

In [5]:
import os
os.listdir()

['.ipynb_checkpoints', 'data', 'project.ipynb']

In [6]:
import pandas as pd
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
db = client["flights_project"]
flights = pd.read_csv("data/flights.csv", low_memory=False, nrows=1000)
flights["date"] = pd.to_datetime(flights[["YEAR", "MONTH", "DAY"]])
flights["year"] = flights["date"].dt.year
flights["month"] = flights["date"].dt.month
db.flights.delete_many({})
db.flights.insert_many(flights.to_dict("records"))
print("flights wrzucone")
airlines = pd.read_csv("data/airlines.csv")
db.airlines.delete_many({})
db.airlines.insert_many(airlines.to_dict("records"))
print("airlines wrzucone")
airports = pd.read_csv("data/airports.csv")
db.airports.delete_many({})
db.airports.insert_many(airports.to_dict("records"))
print("airports wrzucone")

flights wrzucone
airlines wrzucone
airports wrzucone


In [8]:
db.flights.count_documents({})

1000

In [9]:
#1. Pipeline wykorzystuje operator $lookup do połączenia danych lotów z nazwami linii lotniczych, które znajdują się w kolekcji airlines
pipeline = [
    {"$group": {"_id": "$AIRLINE","flights": {"$sum": 1}}},
    {"$lookup": {"from": "airlines","localField": "_id","foreignField": "IATA_CODE","as": "linia"}},
    {"$unwind": "$linia"},
    {"$project": {"_id": 0,"linia": "$linia.AIRLINE","liczba_lotow": "$flights"}},
    {"$sort": {"liczba_lotow": -1}}]
wynik = list(db.flights.aggregate(pipeline))
for x in wynik:
    print(f"a{x['linia']} "f"Liczba lotów: {x['liczba_lotow']}")

aAmerican Airlines Inc. Liczba lotów: 135
aSkywest Airlines Inc. Liczba lotów: 121
aDelta Air Lines Inc. Liczba lotów: 118
aUnited Air Lines Inc. Liczba lotów: 103
aJetBlue Airways Liczba lotów: 91
aAtlantic Southeast Airlines Liczba lotów: 78
aSouthwest Airlines Co. Liczba lotów: 72
aAmerican Eagle Airlines Inc. Liczba lotów: 68
aAlaska Airlines Inc. Liczba lotów: 60
aUS Airways Inc. Liczba lotów: 59
aSpirit Air Lines Liczba lotów: 40
aFrontier Airlines Inc. Liczba lotów: 34
aHawaiian Airlines Inc. Liczba lotów: 17
aVirgin America Liczba lotów: 4


In [10]:
#2. Pipeline oblicza średnie opóźnienie odlotów oraz liczbę lotów dla każdej linii lotniczej
#   Jeśli wartość jest dodatnia - lot był opóźniony, jeśli zaś była ujemna - lot był przyśpieszony
pipeline = [
    {"$match": {"DEPARTURE_DELAY": {"$ne": None}}},
    {"$group": {"_id": "$AIRLINE","avg_delay": {"$avg": "$DEPARTURE_DELAY"},"flights": {"$sum": 1}}},
    {"$sort": {"avg_delay": -1}}]
wynik = list(db.flights.aggregate(pipeline))
for linia in wynik:
    print(f"Linia: {linia['_id']} "f"Średnie opóźnienie: {round(linia['avg_delay'], 2)} minut "f"Liczba lotów: {linia['flights']}")

Linia: NK Średnie opóźnienie: 9.15 minut Liczba lotów: 40
Linia: UA Średnie opóźnienie: 7.81 minut Liczba lotów: 103
Linia: B6 Średnie opóźnienie: 2.71 minut Liczba lotów: 91
Linia: DL Średnie opóźnienie: 2.19 minut Liczba lotów: 118
Linia: US Średnie opóźnienie: 1.12 minut Liczba lotów: 59
Linia: HA Średnie opóźnienie: -2.0 minut Liczba lotów: 17
Linia: VX Średnie opóźnienie: -4.0 minut Liczba lotów: 4
Linia: AS Średnie opóźnienie: nan minut Liczba lotów: 60
Linia: AA Średnie opóźnienie: nan minut Liczba lotów: 135
Linia: OO Średnie opóźnienie: nan minut Liczba lotów: 121
Linia: MQ Średnie opóźnienie: nan minut Liczba lotów: 68
Linia: EV Średnie opóźnienie: nan minut Liczba lotów: 78
Linia: F9 Średnie opóźnienie: nan minut Liczba lotów: 34
Linia: WN Średnie opóźnienie: nan minut Liczba lotów: 72


In [11]:
#3. Pipeline pokazuje ile lotów było każego miesiąca.
#   Tutaj generalnie jest błąd z mojej strony, bo w celu przyspieszenia procesów zapisów dokumentów do kolekcji, ograniczyłem się wyłącznie do 
#   1000 dokumentów. Z tego powodu pokazuje się wyłącznie miesiąc styczeń (oznaczony jako 1) i wypisuje wszystkie dokumenty z kolekcji flights.
#   Aczkolwiek po wrzuceniu tych dokumentów, pipeline będzie działał poprawnie
pipeline = [
    {"$group": {"_id": "$MONTH", "flights": {"$sum": 1}}},
    {"$sort": {"_id": 1}}]
wynik = list(db.flights.aggregate(pipeline))
for x in wynik:
    print(f"Miesiąc: {x['_id']} "f"Liczba lotów: {x['flights']}")

Miesiąc: 1 Liczba lotów: 1000


In [12]:
#4. Pipeline analizuje najczęstsze przyczyny anulowania lotów i podaje ile takich lotów było
#   A - błąd lini lotniczej; B - warunki pogodowe; C - system lotniskowy (zatloczenie itp.)
pipeline = [
    {"$match": {"CANCELLED": 1}},
    {"$group": {"_id": "$CANCELLATION_REASON","flights": {"$sum": 1}}},
    {"$sort": {"flights": -1}}]
wynik = list(db.flights.aggregate(pipeline))
for x in wynik:
    print(f"Powód anulowania: {x['_id']} "f"Liczba lotów: {x['flights']}")

Powód anulowania: B Liczba lotów: 72
Powód anulowania: A Liczba lotów: 11
Powód anulowania: C Liczba lotów: 1


In [13]:
#5. Pipeline oblicza procent lotów punktualnych dla każdej linii lotniczej
pipeline = [
    {"$project": {"AIRLINE": 1,"on_time": {"$cond": [{"$lte": ["$DEPARTURE_DELAY", 0]}, 1, 0]}}},
    {"$group": {"_id": "$AIRLINE","punktualne": {"$sum": "$on_time"},"wszystkie": {"$sum": 1}}},
    {"$project": {"procent_punktualnych": {"$multiply": [{"$divide": ["$punktualne", "$wszystkie"]},100]}}},
    {"$sort": {"procent_punktualnych": -1}}]
wynik = list(db.flights.aggregate(pipeline))
for x in wynik:
    print(f"Linia: {x['_id']} "f"Punktualność: {round(x['procent_punktualnych'],2)}%")

Linia: VX Punktualność: 100.0%
Linia: MQ Punktualność: 92.65%
Linia: AS Punktualność: 90.0%
Linia: HA Punktualność: 88.24%
Linia: B6 Punktualność: 81.32%
Linia: US Punktualność: 79.66%
Linia: AA Punktualność: 78.52%
Linia: DL Punktualność: 77.12%
Linia: WN Punktualność: 76.39%
Linia: EV Punktualność: 74.36%
Linia: OO Punktualność: 73.55%
Linia: F9 Punktualność: 58.82%
Linia: UA Punktualność: 57.28%
Linia: NK Punktualność: 55.0%


In [14]:
#6. Pipeline w biznesowy sposób klasyfikuje loty na punktualne, lekko opóźnione i mocno opóźnione dla lini lotniczej Delta (i tylko dla 20 dokumentów)
pipeline = [
    {"$match": {"AIRLINE": "DL"}},
    {"$project": {"_id": 0,"numer_lotu": "$FLIGHT_NUMBER","data_lotu": {
        "$concat": [{"$toString": "$YEAR"}, "-",{"$toString": "$MONTH"}, "-",{"$toString": "$DAY"}]},
        "kategoria_lotu": {"$cond": [{"$lte": ["$DEPARTURE_DELAY", 0]},"punktualny",
        {"$cond": [{"$lte": ["$DEPARTURE_DELAY", 15]},"lekko opóźniony","mocno opóźniony"]}]}}},
    {"$limit": 20}]
wynik = list(db.flights.aggregate(pipeline))
for x in wynik:
    print(f"Numer lotu: DL{x['numer_lotu']} "f"Data: {x['data_lotu']} "f"Kategoria: {x['kategoria_lotu']}")

Numer lotu: DL806 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL1173 Data: 2015-1-1 Kategoria: lekko opóźniony
Numer lotu: DL2336 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL1434 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL2324 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL2440 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL1560 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL1670 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL1450 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL95 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL2099 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL824 Data: 2015-1-1 Kategoria: mocno opóźniony
Numer lotu: DL941 Data: 2015-1-1 Kategoria: lekko opóźniony
Numer lotu: DL2079 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL1890 Data: 2015-1-1 Kategoria: mocno opóźniony
Numer lotu: DL2453 Data: 2015-1-1 Kategoria: punktualny
Numer lotu: DL421 Data: 2015-1-1 Kategoria: lekko opóźniony
Numer lotu: DL731 Data: 2015-

In [15]:
#Prosty WebScraping, który sprawdza pogodę dla lotniska w Atlantic City
import requests
from bs4 import BeautifulSoup
url = "https://www.timeanddate.com/weather/usa/atlantic-city"
resp = requests.get(url)
soup = BeautifulSoup(resp.text, "html.parser")
temp = soup.find("div", class_="h2").get_text(strip=True)
opis = soup.find("p").get_text(strip=True)
pogoda = f"{temp}, {opis}"
print("Pogoda Atlantic City:", pogoda)

Pogoda Atlantic City: -5 °C, Sunny.


In [16]:
#   Program pobiera dane o lotach i na ich podstawie uczy model ML przewidywać opóźnienie wylotu. 
#   Wykorzystuje regresję liniową oraz dane takie jak linia lotnicza, miesiąc, dzień tygodnia i dystans lotu. 
#   Na końcu sprawdzana jest skuteczność modelu za pomocą średniego błędu bezwzględnego czyli o jaką wartość (tutaj w minutach) model średnio się myli.
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error
pipeline_mongo = [
    {"$match": {"DEPARTURE_DELAY": {"$ne": None}}},
    {"$project": {"AIRLINE": 1,"MONTH": 1,"DAY_OF_WEEK": 1,"DISTANCE": 1,"DEPARTURE_DELAY": 1}}]
dane = list(db.flights.aggregate(pipeline_mongo))
df = pd.DataFrame(dane)
df = df.dropna(subset=["DEPARTURE_DELAY", "MONTH", "DAY_OF_WEEK", "DISTANCE", "AIRLINE"])
X = df[["AIRLINE", "MONTH", "DAY_OF_WEEK", "DISTANCE"]]
y = df["DEPARTURE_DELAY"]
preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["AIRLINE"]),
    ("num", "passthrough", ["MONTH", "DAY_OF_WEEK", "DISTANCE"])])
model = Pipeline([
    ("prep", preprocess),
    ("reg", LinearRegression())])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print("Średni błąd bezwzględny:", round(mean_absolute_error(y_test, pred), 2))

Średni błąd bezwzględny: 14.15
